# Incident simulator: 30 House15 buildings, 30 days


In [1]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PARENT = PROJECT_ROOT.parent
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

from building_maintenance_agents.house_graph.samples import House15Factory
from building_maintenance_agents.house_graph.states import IncidentState
from building_maintenance_agents.incident_simulator import IncidentSimulator, IncidentType
from loguru import logger

logger.disable("building_maintenance_agents")


def ensure_incident_state(house):
    """HouseFactory currently does not initialize incident_state in custom __init__ nodes/edges."""
    for element in [*house.nodes, *house.edges]:
        if not hasattr(element, "incident_state"):
            element.incident_state = IncidentState()
    return house


print("Loaded IncidentSimulator from", PROJECT_ROOT)

Loaded IncidentSimulator from /home/fedor/Projects/building_maintenance_agents


In [2]:
# Scenario parameters. These are the knobs to change first.
N_HOUSES = 30
DAYS = 30
STEPS_PER_DAY = 24
TOTAL_STEPS = DAYS * STEPS_PER_DAY

# Probability scale: 1.0 means use calibrated per-type probabilities from IncidentType.
BASE_INCIDENT_PROBABILITY = 1.0
ENABLE_SPREAD = True
BASE_SEED = 42

PARAMETERS = {
    "n_houses": N_HOUSES,
    "days": DAYS,
    "steps_per_day": STEPS_PER_DAY,
    "total_steps": TOTAL_STEPS,
    "base_incident_probability": BASE_INCIDENT_PROBABILITY,
    "enable_spread": ENABLE_SPREAD,
    "base_seed": BASE_SEED,
    "house_factory": "House15Factory",
}

PARAMETERS

{'n_houses': 30,
 'days': 30,
 'steps_per_day': 24,
 'total_steps': 720,
 'base_incident_probability': 1.0,
 'enable_spread': True,
 'base_seed': 42,
 'house_factory': 'House15Factory'}

In [3]:
# для удобства отображения
def describe_location(simulator, location_id: int) -> dict:
    if location_id in simulator.node_by_id:
        element = simulator.node_by_id[location_id]
        location_kind = "node"
    elif location_id in simulator.edge_by_id:
        element = simulator.edge_by_id[location_id]
        location_kind = "edge"
    else:
        return {
            "location_kind": None,
            "element_type": None,
            "section": None,
            "floor": None,
            "flat_index": None,
        }

    features = getattr(element, "features", {}) or {}
    return {
        "location_kind": location_kind,
        "element_type": type(element).__name__,
        "section": features.get("section"),
        "floor": features.get("floor"),
        "flat_index": features.get("flat_index"),
    }


def to_jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, Counter):
        return dict(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

In [4]:
scenario_events = []
house_summaries = []

progress = tqdm(total=N_HOUSES * DAYS, desc="Simulation", unit="day")

for house_index in range(1, N_HOUSES + 1):
    house = ensure_incident_state(House15Factory.build())
    simulator = IncidentSimulator(
        house,
        base_incident_probability=BASE_INCIDENT_PROBABILITY,
        random_seed=BASE_SEED + house_index,
        enable_spread=ENABLE_SPREAD,
    )

    created_count = 0
    resolved_count = 0
    spread_count = 0
    peak_active = 0
    created_by_type = Counter()

    for day in range(1, DAYS + 1):
        for hour_in_day in range(1, STEPS_PER_DAY + 1):
            step = (day - 1) * STEPS_PER_DAY + hour_in_day
            result = simulator.step()
            peak_active = max(peak_active, len(simulator.active_incidents))

            for incident in result["new_incidents"]:
                created_count += 1
                created_by_type[incident["type"]] += 1
                scenario_events.append({
                    "house_index": house_index,
                    "day": day,
                    "step": step,
                    "event": "new",
                    "incident_id": incident["id"],
                    "incident_type": incident["type"],
                    "severity": float(incident["severity"]),
                    "location_id": incident["location"],
                    **describe_location(simulator, incident["location"]),
                })

            for incident in result["resolved_incidents"]:
                resolved_count += 1
                scenario_events.append({
                    "house_index": house_index,
                    "day": day,
                    "step": step,
                    "event": "resolved",
                    "incident_id": incident["id"],
                    "incident_type": incident["type"],
                    "location_id": incident["location"],
                    **describe_location(simulator, incident["location"]),
                })

            for spread in result["spread_incidents"]:
                for location_id in spread["new_locations"]:
                    spread_count += 1
                    scenario_events.append({
                        "house_index": house_index,
                        "day": day,
                        "step": step,
                        "event": "spread",
                        "source_incident_id": spread["source"],
                        "location_id": location_id,
                        **describe_location(simulator, location_id),
                    })

        progress.update(1)
        progress.set_postfix(
            house=house_index,
            day=day,
            active=len(simulator.active_incidents),
            events=len(scenario_events),
        )

    stats = simulator.get_statistics()
    house_summaries.append({
        "house_index": house_index,
        "nodes": len(house.nodes),
        "edges": len(house.edges),
        "created_incidents": created_count,
        "resolved_incidents": resolved_count,
        "spread_events": spread_count,
        "active_incidents_at_end": len(simulator.active_incidents),
        "peak_active_incidents": peak_active,
        "total_damage": float(stats["total_damage"]),
        "average_active_severity_at_end": float(stats["average_severity"]),
        "created_by_type": dict(created_by_type),
    })

    print(
        f"house {house_index:02d}: created={created_count}, "
        f"spread={spread_count}, resolved={resolved_count}, "
        f"active_end={len(simulator.active_incidents)}, peak={peak_active}"
    )

progress.close()

print(f"Generated {len(scenario_events)} scenario events")

Simulation:   0%|          | 0/900 [00:00<?, ?day/s]

house 01: created=3, spread=1, resolved=4, active_end=0, peak=2
house 02: created=3, spread=5, resolved=8, active_end=0, peak=3
house 03: created=4, spread=2, resolved=6, active_end=0, peak=2
house 04: created=5, spread=6, resolved=11, active_end=0, peak=5
house 05: created=2, spread=0, resolved=1, active_end=1, peak=1
house 06: created=7, spread=1, resolved=8, active_end=0, peak=3
house 07: created=3, spread=2, resolved=5, active_end=0, peak=4
house 08: created=6, spread=1, resolved=7, active_end=0, peak=2
house 09: created=1, spread=0, resolved=1, active_end=0, peak=1
house 10: created=8, spread=9, resolved=17, active_end=0, peak=4
house 11: created=5, spread=5, resolved=10, active_end=0, peak=4
house 12: created=2, spread=0, resolved=2, active_end=0, peak=1
house 13: created=2, spread=0, resolved=2, active_end=0, peak=2
house 14: created=1, spread=0, resolved=1, active_end=0, peak=1
house 15: created=8, spread=3, resolved=11, active_end=0, peak=4
house 16: created=5, spread=2, resol

In [5]:
totals_by_type = Counter()
for summary in house_summaries:
    totals_by_type.update(summary["created_by_type"])

overall_summary = {
    "houses": len(house_summaries),
    "events": len(scenario_events),
    "created_incidents": sum(x["created_incidents"] for x in house_summaries),
    "resolved_incidents": sum(x["resolved_incidents"] for x in house_summaries),
    "spread_events": sum(x["spread_events"] for x in house_summaries),
    "peak_active_incidents_over_all_houses": max(x["peak_active_incidents"] for x in house_summaries),
    "created_by_type": dict(totals_by_type),
}

overall_summary

{'houses': 30,
 'events': 398,
 'created_incidents': 125,
 'resolved_incidents': 194,
 'spread_events': 79,
 'peak_active_incidents_over_all_houses': 6,
 'created_by_type': {'gvs_riser_failure': 77,
  'gvs_pipe_failure': 34,
  'hvs_pipe_failure': 5,
  'hvs_riser_failure': 9}}

In [6]:
# Peek at the first events. Use scenario_events directly for deeper analysis.
scenario_events[:20]

[{'house_index': 1,
  'day': 20,
  'step': 476,
  'event': 'new',
  'incident_id': 0,
  'incident_type': 'gvs_riser_failure',
  'severity': 0.5514799850689387,
  'location_id': 129109153811952,
  'location_kind': 'node',
  'element_type': 'RiserNode',
  'section': 1.0,
  'floor': 9.0,
  'flat_index': 2.0},
 {'house_index': 1,
  'day': 21,
  'step': 481,
  'event': 'new',
  'incident_id': 1,
  'incident_type': 'gvs_pipe_failure',
  'severity': 0.31830371344386776,
  'location_id': 129109154334336,
  'location_kind': 'edge',
  'element_type': 'FlowEdge',
  'section': None,
  'floor': None,
  'flat_index': None},
 {'house_index': 1,
  'day': 21,
  'step': 496,
  'event': 'resolved',
  'incident_id': 1,
  'incident_type': 'gvs_pipe_failure',
  'location_id': 129109154334336,
  'location_kind': 'edge',
  'element_type': 'FlowEdge',
  'section': None,
  'floor': None,
  'flat_index': None},
 {'house_index': 1,
  'day': 21,
  'step': 503,
  'event': 'resolved',
  'incident_id': 0,
  'incident

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "notebooks" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

payload = {
    "parameters": PARAMETERS,
    "overall_summary": overall_summary,
    "house_summaries": house_summaries,
    "events": scenario_events,
}

output_path = RESULTS_DIR / "month_scenarios_30_house15.json"
# with output_path.open("w", encoding="utf-8") as file:
#    json.dump(payload, file, ensure_ascii=False, indent=2, default=to_jsonable)

# print(output_path)



/home/fedor/Projects/building_maintenance_agents/notebooks/results/month_scenarios_30_house15.json
